In [ ]:
import subprocess, sys, os, glob, shutil
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:], flush=True); return r
sh("find /kaggle/input -maxdepth 4 | head -30")
sh("java -version")

In [ ]:
# dựng cây ROOT/harness + ROOT/runs từ dataset (zip tự giải hoặc chưa)
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
src = glob.glob("/kaggle/input/*/ktc-chu-27-10") + glob.glob("/kaggle/input/ktc-chu-27-10") + glob.glob("/kaggle/input/datasets/*/ktc-chu-27-10")
print(src); S = src[0]
for z in glob.glob(S + "/*.zip"): sh(f"cd {W} && unzip -o -q {z} -d {W}/{os.path.basename(z)[:-4]}")
for d in ("harness", "runs"):
    if os.path.isdir(f"{S}/{d}"): shutil.copytree(f"{S}/{d}", f"{W}/{d}", dirs_exist_ok=True)
sh(f"find {W} -type f | head -30")

In [ ]:
sh("(java -version 2>&1 | head -1) || true")
r = sh("java -version 2>&1 | grep -c 'version \"1.8' ")
if r.stdout.strip() != "1":
    sh("apt-get update -qq && apt-get install -y -qq openjdk-8-jre-headless 2>&1 | tail -2")
    sh("ls /usr/lib/jvm")
sh(f"{sys.executable} -m pip install -q pycocoevalcap sacrebleu bert_score rouge_score nltk 2>&1 | tail -2")
import nltk
for p in ("wordnet", "omw-1.4"): nltk.download(p, quiet=True)

In [ ]:
jv = sorted(glob.glob("/usr/lib/jvm/*8*")); print(jv)
if jv: os.environ["JAVA_HOME"] = jv[0]; os.environ["PATH"] = jv[0] + "/bin:" + os.environ["PATH"]
sh("java -version")

In [ ]:
import pycocoevalcap.spice.spice as _sp   # pycocoevalcap là namespace package, không có __file__
L = os.path.join(os.path.dirname(_sp.__file__), "lib"); os.makedirs(L, exist_ok=True)
js = glob.glob("/kaggle/input/**/stanford-corenlp-3.6.0*.jar", recursive=True); print(js)
assert len(js) >= 2, "thiếu jar CoreNLP"
for j in js: shutil.copy(j, L)
sh(f"ls -la {L} | grep corenlp")

In [ ]:
# harness/ktc_chu_27_10.py nằm ở W/harness; ROOT = W
import subprocess
p = subprocess.Popen(f"cd {W}/harness && {sys.executable} -u ktc_chu_27_10.py --B 2000 > {W}/ktc.log 2>&1", shell=True)
import time
while p.poll() is None:
    time.sleep(120); print(time.strftime("%H:%M"), open(f"{W}/ktc.log").read()[-400:].replace("\n", " | "), flush=True)
print(open(f"{W}/ktc.log").read()[-3000:])

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/ktc_chu_27_10.zip", "w") as z:
    for f in ("ktc.log",): z.write(f"{W}/{f}", f)
    for f in glob.glob(f"{W}/runs/ktc_chu_27_10.json"): z.write(f, os.path.basename(f))
shutil.copy(f"{W}/ktc.log", "/kaggle/working/ktc.log")
for f in glob.glob(f"{W}/runs/ktc_chu_27_10.json"): shutil.copy(f, "/kaggle/working/")
shutil.rmtree(W)
print(os.listdir("/kaggle/working"))